In [ ]:
# Import all basic functions
import numpy as np
import h5py
import torch

# Import config
from config import load_config

import tqdm

DEVICE = torch.device("cuda")


In [ ]:
# data generation
from generate_data.datagen import (
    knn_gpu,
    estimate_local_tangent_space_p_var,
)


# Load the perturbed points and sample the tangent space at each one

We load the already-computed perturbed points from `perturbed_sweep.h5`
(produced by `generate_manifold.ipynb`). For a chosen set of layers, we:

1. Load that layer's points onto the GPU.
2. Find each point's k nearest neighbors (`k = d_model + 1`) with `knn_gpu`.
3. Estimate the local tangent space at each point from its neighborhood
   (`estimate_local_tangent_space_p_var`).
4. Sample 1000 points from a ball in that tangent subspace, centered at the
   point.

We deliberately do **not** persist the tangent bases or the sampled points to
disk. A full tangent basis is `(d_model, d_model)` floats per point -- across
hundreds of thousands of points and many layers that's already ~300GB per
config in `local_tangent_spaces.h5`, and saving 1000 samples per point on top
of that would be tens of terabytes (far more than the ~2.6TB free on disk).
So both the basis and the samples are computed on the fly, per point, and
discarded once we move to the next point -- this notebook is for inspecting
the tangent-space samples in memory (e.g. one point/batch at a time), not for
building a saved dataset.


In [ ]:
# Load config
cfg = load_config("configs/smoll.toml")

perturbed_h5_path = "onnx_models/smoll/perturbed_sweep.h5"

n_samples_per_point = 1000  # points to sample from the tangent ball at each point
tangent_radius = 0.1  # ball radius in the tangent subspace
p_var = 1  # variance fraction for estimate_local_tangent_space_p_var (1 => full local rank)

# Network-depth order of layer output names (this is the order the model
# actually produces them in -- HDF5 group iteration order in
# perturbed_sweep.h5 does NOT match this, e.g. it yields "add_102" before
# "add_96", so we must not slice h5f.keys() directly).
depth_ordered_layers = [s.output for s in cfg.sublayers]

with h5py.File(perturbed_h5_path, "r") as h5f:
    available = set(h5f.keys())
all_layer_names = [name for name in depth_ordered_layers if name in available]

# Subsample layers: 2 from the start, 2 from the middle, 2 from the end,
# instead of all of them -- keeps this exploratory pass cheap while still
# covering early/mid/late representations.
n_layers_total = len(all_layer_names)
mid = n_layers_total // 2
layer_names = list(
    dict.fromkeys(
        all_layer_names[0:2]
        + all_layer_names[mid : mid + 2]
        + all_layer_names[-2:]
    )
)

print(f"{n_layers_total} layers available, sampling from: {layer_names}")


In [ ]:
def sample_ball_in_subspace_gpu(
    basis: torch.Tensor,
    center: torch.Tensor,
    radius: float,
    n_samples: int,
    generator: torch.Generator | None = None,
) -> torch.Tensor:
    """GPU/torch counterpart of `sample_ball_in_subspace` in generate_data.datagen.

    Samples `n_samples` points uniformly (by volume) from the d-dimensional
    ball of the given `radius`, embedded in the subspace spanned by the rows
    of `basis` (need not be orthonormal), centered at `center`. Kept local to
    this notebook (rather than reusing the numpy version) so the whole
    per-point loop stays on the GPU -- with hundreds of thousands of points
    per layer, round-tripping each basis/center to CPU would dominate runtime.

    basis: (d, D) tensor -- d basis row-vectors in the D-dim ambient space.
    center: (D,) tensor.
    Returns: (n_samples, D) tensor on the same device/dtype as `basis`.
    """
    device, dtype = basis.device, basis.dtype
    d, D = basis.shape

    # Orthonormal basis for the subspace (rows of basis -> orthonormal rows of Q).
    Q, _ = torch.linalg.qr(basis.T)  # (D, d), orthonormal columns
    Q = Q.T  # (d, D)

    # Uniform sampling in a d-ball: normalize a Gaussian vector for direction,
    # then scale radius by u^(1/d) for correct volume density.
    g = torch.randn((n_samples, d), device=device, dtype=dtype, generator=generator)
    directions = g / g.norm(dim=-1, keepdim=True)
    u = torch.rand((n_samples, 1), device=device, dtype=dtype, generator=generator)
    radii = radius * u.pow(1.0 / d)
    local_points = directions * radii  # (n_samples, d)

    return center.unsqueeze(0) + local_points @ Q  # (n_samples, D)


In [ ]:
# Per layer: load points, find neighbors, then for each point compute its
# tangent space and sample n_samples_per_point from it -- all in memory.
# Nothing is written to disk here; `last_*` just holds the most recent
# point's results so they can be inspected below.
generator = torch.Generator(device=DEVICE).manual_seed(0)

last_samples = None
last_tangent_basis = None
last_center = None
last_k = None

with h5py.File(perturbed_h5_path, "r") as h5f:
    for layer_name in layer_names:
        grp = h5f[layer_name]
        flat = torch.from_numpy(grp["vec"][:]).to(DEVICE)
        d_model = flat.shape[1]
        n_points = flat.shape[0]

        kns = knn_gpu(flat, k=d_model + 1)

        for j in tqdm.tqdm(range(n_points), desc=layer_name):
            neighbor_row_idx = kns[1][j]
            tangent_basis, k = estimate_local_tangent_space_p_var(
                flat[neighbor_row_idx], p=p_var
            )

            samples = sample_ball_in_subspace_gpu(
                tangent_basis,
                flat[j],
                radius=tangent_radius,
                n_samples=n_samples_per_point,
                generator=generator,
            )

            # Discard everything except the most recent point's results --
            # see the markdown cell above for why we don't accumulate/save.
            last_samples = samples
            last_tangent_basis = tangent_basis
            last_center = flat[j]
            last_k = int(k)

        del flat, kns
        torch.cuda.empty_cache()

print("done -- last point's tangent dim:", last_k, "samples shape:", last_samples.shape)


In [ ]:
# Sanity check on the last point processed: samples should lie within
# tangent_radius of the center (up to floating point slop), and the mean of
# the samples should roughly recover the center itself.
dists_from_center = (last_samples - last_center.unsqueeze(0)).norm(dim=-1)

print("tangent dim (k):", last_k)
print("samples shape:", tuple(last_samples.shape))
print("max distance from center:", dists_from_center.max().item(), "(radius =", tangent_radius, ")")
print("mean of samples vs center, max abs diff:", (last_samples.mean(dim=0) - last_center).abs().max().item())
